# 라이브러리·모델 사전 다운로드 (16기 AI 챌린지 데스크탑)

대회 당일 1,000명이 동시에 라이브러리와 모델을 다운로드하면 부하가 몰립니다.
대회 시작 전에 이 노트북을 실행해 **베이스라인에서 사용하는 라이브러리(wheel)와 사전학습 모델을 로컬에 미리 다운로드**합니다.

- 다운로드 대상 : `(260827)_baseline_desktop5060ti.ipynb` 에서 사용하는 라이브러리와 모델 `Qwen/Qwen2.5-VL-3B-Instruct`
- 저장 위치 : 이 노트북과 같은 폴더에 `downloads` 폴더가 생성되고 그 안에 저장됩니다.
- 데이터셋 : 별도 배포되어 `data` 폴더에 준비합니다. **이 노트북은 데이터를 다운로드하지 않습니다.**
- 용량·시간 : 총 약 11GB(라이브러리 약 3.5GB, 모델 약 7.5GB). 네트워크 속도에 따라 20~40분 정도 소요됩니다.

```
(260902) 라이브러리, 모델 다운로드/
├─ (260902)_download_libs_models.ipynb     ← 사전 다운로드 노트북 (대회 전 실행)
├─ (260902)_baseline_desktop5060ti_offline.ipynb   ← 오프라인 베이스라인 (대회 당일 실행)
├─ data/                                   ← 사전 배포 데이터셋 (직접 준비)
│  ├─ train/ , test/
│  └─ train.csv , test.csv , sample_submission.csv
└─ downloads/                              ← 다운로드 노트북이 생성
   ├─ libs/                                ← 라이브러리 wheel 파일
   └─ models/Qwen2.5-VL-3B-Instruct/       ← 사전학습 모델
```


# 환경 준비

기존 데스크탑 환경설정 가이드와 동일하게 진행합니다.

- 터미널에서 가상환경 생성 : `py -3.11 -m venv baseline` (각자 파이썬 버전에 맞게)
- 우측 상단 Select Kernel - Python Environment - **baseline** 선택
- 아래 셀 실행 → ipykernel 설치 → hello 출력 확인

> 대회 당일 오프라인 베이스라인도 **같은 가상환경(baseline)** 에서 실행합니다. 가상환경을 삭제하지 마세요.


In [5]:
print('hello123')

hello123


In [6]:
import os, sys, platform

# 파이썬, 가상환경 확인 : 다운로드되는 wheel 파일은 이 파이썬 버전(3.11~3.13)·64bit 기준으로 결정됩니다.
print("Python   :", sys.version.split()[0], "|", platform.architecture()[0])
print("실행 파일 :", sys.executable)
print("가상환경  :", "예" if sys.prefix != sys.base_prefix else "아니오 → 상단 Select Kernel 에서 baseline 가상환경을 선택하세요")

# 조건에 맞지 않으면 여기서 멈춤 (뒤 셀에서 알 수 없는 오류로 이어지는 것을 방지)
assert platform.architecture()[0] == "64bit", "64bit 파이썬이 필요합니다. 환경설정 가이드대로 Windows installer (64-bit) 를 설치하세요."
assert (3, 10) <= sys.version_info[:2] <= (3, 14), f"Python 3.11~3.13 이 필요합니다 (현재 {sys.version.split()[0]})."


Python   : 3.11.9 | 64bit
실행 파일 : c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Scripts\python.exe
가상환경  : 예


In [7]:
# 저장 폴더 생성
ASSET_DIR = "downloads"                                   # 새로 생성되는 폴더
LIB_DIR   = os.path.join(ASSET_DIR, "libs")                 # 라이브러리 wheel 파일
MODEL_DIR = os.path.join(ASSET_DIR, "models")               # 사전학습 모델

MODEL_ID        = "Qwen/Qwen2.5-VL-3B-Instruct"
MODEL_LOCAL_DIR = os.path.join(MODEL_DIR, MODEL_ID.split("/")[-1])   # downloads/models/Qwen2.5-VL-3B-Instruct

os.makedirs(LIB_DIR, exist_ok=True)
os.makedirs(MODEL_LOCAL_DIR, exist_ok=True)

print("라이브러리 저장 :", os.path.abspath(LIB_DIR))
print("모델 저장     :", os.path.abspath(MODEL_LOCAL_DIR))


라이브러리 저장 : c:\Users\SSAFY\Desktop\AI2_Challenge\downloads\libs
모델 저장     : c:\Users\SSAFY\Desktop\AI2_Challenge\downloads\models\Qwen2.5-VL-3B-Instruct


# 1. 라이브러리 다운로드

`pip download` 는 패키지를 **설치하지 않고 wheel 파일만 저장**합니다. 한 번의 명령으로 모든 라이브러리를 함께 받습니다.

- torch / torchvision / torchaudio : PyTorch 정식 **CUDA 12.8 빌드** `2.11.0+cu128` 로 버전 고정 (RTX 5060 Ti 지원)
- transformers, accelerate, peft, bitsandbytes, datasets, pillow, pandas : 베이스라인 라이브러리
- ipykernel : 대회 당일 새 가상환경에서 커널을 만들 경우를 대비해 함께 저장

> torch 버전을 `+cu128` 까지 고정하는 이유 : transformers, peft 등이 torch 에 의존하므로, 고정하지 않으면 pip 이 PyPI 의 **CPU 전용 torch** 를 함께 받아 오고 설치 시 그쪽이 선택될 수 있습니다.
>
> **가장 큰 torch 파일이 약 3GB 라서 5~20분 걸립니다.** 셀 왼쪽 표시가 완료로 바뀌고 마지막 줄에 `Successfully downloaded ...` 와 `라이브러리 다운로드 완료` 가 출력될 때까지 기다려 주세요.
>
> pip 은 **모든 파일을 받은 뒤 마지막에 한 번에** `downloads/libs` 로 옮깁니다. 중간에 멈추거나 오류가 나면 폴더가 비어 있으므로, 출력의 `ERROR` 줄을 확인한 뒤 **같은 셀을 다시 실행**하세요. 이미 받은 부분은 pip 캐시에 남아 있어 두 번째는 빠르게 끝납니다.


In [8]:
# torch(cu128) + 베이스라인 라이브러리 + ipykernel 을 한 번에 다운로드 (약 3.5GB, 5~20분)
%pip download torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchaudio==2.11.0+cu128 "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas ipykernel --extra-index-url https://download.pytorch.org/whl/cu128 -d downloads/libs

# 다운로드가 실제로 완료되었는지 확인 : 실패했으면 여기서 멈추고 안내를 출력
import glob
wheels = glob.glob(os.path.join(LIB_DIR, "*.whl"))
torch_ok = any(os.path.basename(w).startswith("torch-2.11.0+cu128") for w in wheels)
hf_ok    = any(os.path.basename(w).startswith("huggingface_hub-") for w in wheels)
if not (torch_ok and hf_ok):
    raise SystemExit(
        f"[다운로드 미완료] {LIB_DIR} 에 wheel 파일 {len(wheels)}개. "
        "위 출력에서 ERROR 줄을 확인하고 (네트워크 끊김, 디스크 용량 부족 등) 이 셀을 다시 실행하세요. "
        "완료 전에는 다음 셀로 넘어가지 마세요."
    )
print(f"라이브러리 다운로드 완료 : wheel {len(wheels)}개")

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\torch-2.11.0+cu128-cp311-cp311-win_amd64.whl
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\torchvision-0.26.0+cu128-cp311-cp311-win_amd64.whl
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\torchaudio-2.11.0+cu128-cp311-cp311-win_amd64.whl
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\transformers-4.57.6-py3-none-any.whl
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\accelerate-1.15.0-py3-none-any.whl
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\peft-0.20.0-py3-none-any.whl
  File was already downloaded c:\users\ssafy\desktop\ai2_challenge\downloads\libs\bitsandbytes-0.50.2-py3-none-win_amd64.whl
  File was already downloaded c:\users\ssafy\desktop


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. 모델 다운로드

Hugging Face 의 `{MODEL_ID}` 저장소 전체를 `downloads/models/Qwen2.5-VL-3B-Instruct` 폴더에 그대로 저장합니다(약 7.5GB, safetensors 2개 파일).

- 모델 다운로드 도구 `huggingface_hub` 를 먼저 설치합니다. 1장에서 받은 wheel 이 있으면 그것을 쓰고, 없으면 PyPI 에서 받습니다(이 노트북은 인터넷 연결 상태에서 실행합니다).
- 모델 파일은 받는 즉시 폴더에 저장되므로, 중간에 끊긴 경우 같은 셀을 다시 실행하면 받은 부분은 건너뛰고 이어서 진행됩니다.


In [9]:
# huggingface_hub 설치 : downloads/libs 의 wheel 을 우선 사용, 없으면 PyPI 에서 설치
%pip -q install --find-links downloads/libs "huggingface_hub<1.0"

import importlib
try:
    hf = importlib.import_module("huggingface_hub")
except ModuleNotFoundError:
    raise SystemExit("huggingface_hub 설치 실패. 위 출력의 ERROR 줄을 확인하세요. 인터넷 연결 상태에서 이 셀을 다시 실행하세요.")
print("huggingface_hub", hf.__version__, "설치 확인")

Note: you may need to restart the kernel to use updated packages.
huggingface_hub 0.36.2 설치 확인



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=MODEL_LOCAL_DIR,
    ignore_patterns=["*.md", ".gitattributes"],   # 문서 파일 제외
    max_workers=4,
)
print("모델 저장 위치:", os.path.abspath(local_path))


c:\Users\SSAFY\Desktop\AI2_Challenge\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|██████████| 12/12 [00:32<00:00,  2.74s/it]

모델 저장 위치: C:\Users\SSAFY\Desktop\AI2_Challenge\downloads\models\Qwen2.5-VL-3B-Instruct


# 3. 다운로드 확인

라이브러리 wheel 파일과 모델 파일이 모두 있는지 확인합니다. `누락` 이 출력되면 해당 단계 셀을 다시 실행하세요.


In [11]:
import glob, json

def size_gb(paths):
    return sum(os.path.getsize(p) for p in paths) / 1024**3

# --- 라이브러리 ---
wheels = glob.glob(os.path.join(LIB_DIR, "*.whl")) + glob.glob(os.path.join(LIB_DIR, "*.tar.gz"))
print(f"[라이브러리] {len(wheels)}개 파일, {size_gb(wheels):.2f} GB  ({os.path.abspath(LIB_DIR)})")
names = [os.path.basename(p).lower() for p in wheels]
for pkg in ["torch", "torchvision", "torchaudio", "transformers", "accelerate", "peft",
            "bitsandbytes", "datasets", "pillow", "pandas", "huggingface_hub", "ipykernel"]:
    hit = [n for n in names if n.startswith(pkg.replace("-", "_") + "-")]
    print(f"  {'OK  ' if hit else '누락'} {pkg:<16s} {', '.join(hit)}")

# torch 계열은 반드시 cu128 빌드 1개씩만 있어야 함 (CPU 빌드가 섞이면 설치 시 그쪽이 선택될 수 있음)
for pkg in ["torch", "torchvision", "torchaudio"]:
    hit = [n for n in names if n.startswith(pkg + "-")]
    bad = [n for n in hit if "+cu128" not in n]
    if bad:
        print(f"  경고 {pkg}: CPU 빌드 wheel 이 섞여 있습니다 → 삭제하세요: {bad}")
    elif len(hit) != 1:
        print(f"  경고 {pkg}: cu128 wheel 이 {len(hit)}개 입니다 (1개여야 함)")

# --- 모델 ---
print()
model_files = glob.glob(os.path.join(MODEL_LOCAL_DIR, "*"))
model_files = [p for p in model_files if os.path.isfile(p)]
print(f"[모델] {len(model_files)}개 파일, {size_gb(model_files):.2f} GB  ({os.path.abspath(MODEL_LOCAL_DIR)})")
required = ["config.json", "generation_config.json", "preprocessor_config.json", "chat_template.json",
            "tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt", "model.safetensors.index.json"]
index_path = os.path.join(MODEL_LOCAL_DIR, "model.safetensors.index.json")
if os.path.exists(index_path):
    with open(index_path, encoding="utf-8") as f:
        required += sorted(set(json.load(f)["weight_map"].values()))   # safetensors 샤드 파일
for fn in required:
    p = os.path.join(MODEL_LOCAL_DIR, fn)
    ok = os.path.exists(p)
    print(f"  {'OK  ' if ok else '누락'} {fn:<36s} {os.path.getsize(p)/1024**2:>10.1f} MB" if ok else f"  누락 {fn}")


[라이브러리] 75개 파일, 2.71 GB  (c:\Users\SSAFY\Desktop\AI2_Challenge\downloads\libs)
  OK   torch            torch-2.11.0+cu128-cp311-cp311-win_amd64.whl
  OK   torchvision      torchvision-0.26.0+cu128-cp311-cp311-win_amd64.whl
  OK   torchaudio       torchaudio-2.11.0+cu128-cp311-cp311-win_amd64.whl
  OK   transformers     transformers-4.57.6-py3-none-any.whl
  OK   accelerate       accelerate-1.15.0-py3-none-any.whl
  OK   peft             peft-0.20.0-py3-none-any.whl
  OK   bitsandbytes     bitsandbytes-0.50.2-py3-none-win_amd64.whl
  OK   datasets         datasets-5.0.1-py3-none-any.whl
  OK   pillow           pillow-12.3.0-cp311-cp311-win_amd64.whl
  OK   pandas           pandas-3.0.5-cp311-cp311-win_amd64.whl
  OK   huggingface_hub  huggingface_hub-0.36.2-py3-none-any.whl
  OK   ipykernel        ipykernel-7.3.0-py3-none-any.whl

[모델] 12개 파일, 7.00 GB  (c:\Users\SSAFY\Desktop\AI2_Challenge\downloads\models\Qwen2.5-VL-3B-Instruct)
  OK   config.json                                 0.0 MB

# 4. 오프라인 설치 검증 (권장)

대회 당일 베이스라인이 실행하는 것과 같은 방식으로, **인터넷 없이 `downloads/libs` 폴더의 wheel 파일만으로** 설치가 되는지 미리 확인합니다.
누락된 의존성이 있으면 여기서 오류가 나므로, 인터넷이 가능한 지금 바로잡을 수 있습니다.

- 설치가 끝나면 torch 버전(`cu128`), `cuda.is_available : True`, `RTX 5060 Ti` 가 출력되어야 합니다.
- 마지막 셀은 로컬 모델 폴더의 설정·프로세서만 오프라인으로 로드해 봅니다(가중치는 로드하지 않아 빠르게 끝납니다).


In [12]:
%pip install --no-index --find-links downloads/libs torch==2.11.0+cu128 torchvision==0.26.0+cu128 torchaudio==2.11.0+cu128

Looking in links: downloads/libs
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\torch-2.11.0+cu128-cp311-cp311-win_amd64.whl
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\torchvision-0.26.0+cu128-cp311-cp311-win_amd64.whl
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\torchaudio-2.11.0+cu128-cp311-cp311-win_amd64.whl
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\sympy-1.14.0-py3-none-any.whl (from torch==2.11.0+cu128)
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\networkx-3.6.1-py3-none-any.whl (from torch==2.11.0+cu128)
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\jinja2-3.1.6-py3-none-any.whl (from torch==2.11.0+cu128)
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\numpy-2.4.6-cp311-cp311-win_amd64.whl (from torchvision==0.26.0+cu128)
Processing c:\users\ssafy\desktop\ai2_challenge\downloads\libs\pillow-12.3.0-cp311-cp311-win_amd64.whl (from torchvision==0.26.0+cu128)


In [13]:
%pip -q install --no-index --find-links downloads/libs "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

Note: you may need to restart the kernel to use updated packages.


In [14]:
# Hugging Face 오프라인 모드로 로컬 모델 폴더 로드 확인
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
print("torch :", torch.__version__)
assert "+cu" in torch.__version__, f"CPU 전용 torch({torch.__version__})가 설치되었습니다. 3장 확인 결과의 경고를 해결한 뒤 위 설치 셀을 다시 실행하세요."
print("cuda.is_available :", torch.cuda.is_available())
print("device_name :", torch.cuda.get_device_name() if torch.cuda.is_available() else "-")

from transformers import AutoConfig, AutoProcessor

config = AutoConfig.from_pretrained(MODEL_LOCAL_DIR, local_files_only=True, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(MODEL_LOCAL_DIR, local_files_only=True, trust_remote_code=True)
print("모델 설정 :", config.model_type, "| 프로세서 :", type(processor).__name__)
print("오프라인 로드 확인 완료")


torch : 2.11.0+cu128
cuda.is_available : True
device_name : NVIDIA GeForce RTX 5060 Ti


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


모델 설정 : qwen2_5_vl | 프로세서 : Qwen2_5_VLProcessor
오프라인 로드 확인 완료


# 완료

- `downloads/libs`, `downloads/models/Qwen2.5-VL-3B-Instruct` 폴더가 준비되었습니다. **폴더를 이동하거나 이름을 바꾸지 마세요.**
- 데이터셋은 배포받은 대로 `data` 폴더(train/, test/, csv 파일)에 준비합니다.
- 대회 당일에는 `(260902)_baseline_desktop5060ti_offline.ipynb` 을 같은 가상환경(baseline)에서 실행합니다.
